# Results: final comparison (#14)

Reads the committed `results/<run>/eval_*/` files and builds the main comparison table and the figures used in the
report and slides. Nothing here trains or re-evaluates a model, so it runs in seconds.

Ground rules, same as the rest of the project:
- All models are trained on FER-2013 only. RAF-DB is never used for training or for choosing a config.
- "Final" models (`cnn_best`, `resnet50`, `vit`) were picked on FER-2013 validation macro-F1.
- Subgroup gap = best minus worst group accuracy on RAF-DB, in points (gender is female vs male, the small
  "unsure" group is left out). One seed per run, so small differences between models can be noise.

In [1]:
# SETUP + HELPERS: ROOT / sys.path as in the other notebooks. RUNS = every run to list; FINAL = the three configs picked on FER-2013
# validation macro-F1 (cnn_best, resnet50, vit); PLOT = models drawn in the figures.
# It imports gap_table / load_audit from src.bias_report, so the fairness numbers here are computed by the same code as the report.
# load_metrics reads a run's metrics.json (written by src.evaluate); balanced_acc = mean per-class recall.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():  # opened from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.bias_report import gap_table, load_audit  # only needs pandas, so no torch here

RESULTS = ROOT / "results"
FIG = RESULTS / "figures"
FIG.mkdir(exist_ok=True)

# order used in the tables; FINAL are the configs picked on FER-2013 val macro-F1
RUNS = ["cnn", "cnn_best", "resnet", "resnet_224", "resnet_frozen", "resnet50", "vit_small", "vit"]
FINAL = ["cnn_best", "resnet50", "vit"]
LABELS = {"cnn": "CNN baseline", "cnn_best": "CNN (tuned)", "resnet": "ResNet-18", "resnet_224": "ResNet-18 224px",
          "resnet_frozen": "ResNet-18 frozen", "resnet50": "ResNet-50", "vit_small": "ViT-S baseline", "vit": "ViT-B"}
PLOT = ["cnn_best", "resnet50", "vit_small", "vit"]  # models shown in the figures
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]  # one colour per plotted model


# INPUT: run name and dataset.  RETURNS: the saved metrics.json as a dict.
def load_metrics(run, dataset):
    with open(RESULTS / run / f"eval_{dataset}" / "metrics.json") as f:
        return json.load(f)


# INPUT: a metrics dict.  RETURNS: mean per-class recall (not inflated by RAF-DB being 39% happy).
def balanced_acc(m):
    return m["per_class"]["macro avg"]["recall"]  # mean per-class recall

## Accuracy and cross-dataset drop

In [2]:
# BUILD THE MAIN TABLE: for every run read FER-2013 and RAF-DB metrics plus the best validation macro-F1 from history.csv,
# then add the 'drop' columns (FER-2013 score minus RAF-DB score) that measure how well each model transfers.
rows = []
for r in RUNS:
    f, a = load_metrics(r, "fer2013"), load_metrics(r, "rafdb")
    rows.append({
        "run": r, "model": LABELS[r],
        "val_macro_f1": pd.read_csv(RESULTS / r / "history.csv")["val_macro_f1"].max(),
        "fer_acc": f["acc"], "fer_macro_f1": f["macro_f1"], "fer_bal_acc": balanced_acc(f),
        "raf_acc": a["acc"], "raf_macro_f1": a["macro_f1"], "raf_bal_acc": balanced_acc(a),
    })
overall = pd.DataFrame(rows)
for k in ("acc", "macro_f1", "bal_acc"):
    overall[f"{k}_drop"] = overall[f"fer_{k}"] - overall[f"raf_{k}"]
overall["final"] = overall["run"].isin(FINAL)

cols = ["model", "val_macro_f1", "fer_acc", "fer_macro_f1", "raf_acc", "raf_macro_f1",
        "acc_drop", "macro_f1_drop", "bal_acc_drop"]
overall[cols].round(3)

,model,val_macro_f1,fer_acc,fer_macro_f1,raf_acc,raf_macro_f1,acc_drop,macro_f1_drop,bal_acc_drop
0,CNN baseline,0.653,0.676,0.648,0.651,0.480,0.025,0.167,0.165
1,CNN (tuned),0.676,0.702,0.684,0.612,0.443,0.090,0.241,0.217
2,ResNet-18,0.672,0.691,0.664,0.604,0.449,0.087,0.215,0.204
3,ResNet-18 224px,0.659,0.703,0.675,0.613,0.465,0.090,0.210,0.196
4,ResNet-18 frozen,0.324,0.345,0.316,0.365,0.233,-0.020,0.083,0.129
5,ResNet-50,0.681,0.701,0.687,0.642,0.493,0.059,0.194,0.152
6,ViT-S baseline,0.649,0.701,0.661,0.628,0.475,0.074,0.186,0.184
7,ViT-B,0.679,0.707,0.679,0.647,0.490,0.060,0.189,0.174


Balanced accuracy is the mean per-class recall. RAF-DB is 39% happy, so plain accuracy flatters every model there;
macro-F1 and balanced accuracy are the better guides to transfer.

## Subgroup gaps on RAF-DB

In [3]:
# FAIRNESS TABLE: stack every run's subgroup audit, compute the best-minus-worst gap, its p-value and whether the confidence
# intervals are separated, then pivot so each model is one row and gender / age / race are columns.
audit = pd.concat([load_audit(RESULTS / r, "rafdb") for r in RUNS])
gaps = gap_table(audit)
gap_pts = gaps.pivot(index="model", columns="attribute", values="gap") * 100
gap_p = gaps.pivot(index="model", columns="attribute", values="p_value")
gap_sep = gaps.pivot(index="model", columns="attribute", values="ci_separated")

show = gap_pts.round(1).add_suffix(" gap (pts)").join(gap_p.round(3).add_suffix(" p"))
show.loc[RUNS]

attribute,age gap (pts),gender gap (pts),race gap (pts),age p,gender p,race p
model,,,,,,
cnn,21.2,8.6,3.9,0.000,0.000,0.099
cnn_best,15.2,10.4,5.3,0.005,0.000,0.114
resnet,18.2,9.0,3.1,0.001,0.000,0.421
resnet_224,9.2,8.2,7.1,0.092,0.000,0.065
resnet_frozen,19.4,9.8,4.5,0.000,0.000,0.173
resnet50,11.7,8.1,4.7,0.033,0.000,0.216
vit_small,14.6,5.4,2.5,0.007,0.003,0.523
vit,12.7,7.3,7.6,0.016,0.000,0.043


`p` is a two-proportion z-test between the best and worst group. Neither it nor the interval check accounts for the
best and worst groups being picked out of several, so both flatter a gap a little. We did not test whether gaps
differ between models.

## Final comparison table

In [4]:
# FINAL TABLE: join the transfer table and the gap table and save it as results/final_comparison.csv (the README links to it).
final = overall.merge(gap_pts.add_suffix("_gap"), left_on="run", right_index=True)
final = final.merge(gap_p.add_suffix("_p"), left_on="run", right_index=True)
final = final.set_index("run").loc[RUNS].reset_index()
final.to_csv(RESULTS / "final_comparison.csv", index=False)

main_cols = ["model", "final", "val_macro_f1", "fer_acc", "fer_macro_f1", "raf_acc", "raf_macro_f1",
             "macro_f1_drop", "bal_acc_drop", "gender_gap", "age_gap", "race_gap"]
final[main_cols].round(3)

,model,final,val_macro_f1,fer_acc,fer_macro_f1,raf_acc,raf_macro_f1,macro_f1_drop,bal_acc_drop,gender_gap,age_gap,race_gap
0,CNN baseline,False,0.653,0.676,0.648,0.651,0.480,0.167,0.165,8.587,21.249,3.925
1,CNN (tuned),True,0.676,0.702,0.684,0.612,0.443,0.241,0.217,10.410,15.203,5.285
2,ResNet-18,False,0.672,0.691,0.664,0.604,0.449,0.215,0.204,9.002,18.162,3.111
3,ResNet-18 224px,False,0.659,0.703,0.675,0.613,0.465,0.210,0.196,8.155,9.230,7.138
4,ResNet-18 frozen,False,0.324,0.345,0.316,0.365,0.233,0.083,0.129,9.845,19.375,4.484
5,ResNet-50,True,0.681,0.701,0.687,0.642,0.493,0.194,0.152,8.062,11.677,4.688
6,ViT-S baseline,False,0.649,0.701,0.661,0.628,0.475,0.186,0.184,5.352,14.586,2.463
7,ViT-B,True,0.679,0.707,0.679,0.647,0.490,0.189,0.174,7.286,12.672,7.639


## Figures

In [5]:
# FIGURE 1: grouped bars of accuracy and macro-F1 on FER-2013 vs RAF-DB for the plotted models.
labels = [LABELS[r] for r in PLOT]
colors = SERIES[:2]
idx = overall.set_index("run")
x = range(len(PLOT))
w = 0.38

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, fer, raf) in zip(axes, [("Accuracy", "fer_acc", "raf_acc"), ("Macro-F1", "fer_macro_f1", "raf_macro_f1")]):
    f = [idx.loc[r, fer] for r in PLOT]
    r_ = [idx.loc[r, raf] for r in PLOT]
    b1 = ax.bar([i - w / 2 for i in x], f, w, label="FER-2013 test", color=colors[0])
    b2 = ax.bar([i + w / 2 for i in x], r_, w, label="RAF-DB test", color=colors[1])
    ax.bar_label(b1, fmt="%.3f", fontsize=8)
    ax.bar_label(b2, fmt="%.3f", fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title)
    ax.set_ylim(0, 0.95)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, ncol=2, loc="upper center")
fig.suptitle("Trained on FER-2013, tested in-domain and on RAF-DB", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "fer_vs_rafdb.png", dpi=200, bbox_inches="tight")
plt.show()

/var/folders/dj/9ms80p1d7_xfsf3my3s8tpl00000gn/T/ipykernel_49864/12295931.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# FIGURE 2: subgroup gaps per attribute for each plotted model; a * marks gaps whose 95% intervals do not overlap.
attrs = ["gender", "age", "race"]
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.2
for i, r in enumerate(PLOT):
    vals = [gap_pts.loc[r, a] for a in attrs]
    bars = ax.bar([j + (i - 1.5) * w for j in range(len(attrs))], vals, w, label=LABELS[r], color=SERIES[i])
    for b, a in zip(bars, attrs):
        star = "*" if gap_sep.loc[r, a] else ""
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.2, f"{b.get_height():.1f}{star}",
                ha="center", fontsize=8)
ax.set_xticks(range(len(attrs)))
ax.set_xticklabels([a.capitalize() for a in attrs])
ax.set_ylabel("best minus worst group accuracy (points)")
ax.set_title("RAF-DB subgroup gaps (* = 95% intervals do not overlap)")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "subgroup_gaps.png", dpi=200, bbox_inches="tight")
plt.show()

/var/folders/dj/9ms80p1d7_xfsf3my3s8tpl00000gn/T/ipykernel_49864/2660211900.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ViT tuning (#8)

In [7]:
# FIGURE 3: validation macro-F1 of every ViT tuning run (the selected one, ViT-B, is drawn in a different colour).
vit_runs = {"vit_small_lr1e5": "ViT-S 1e-5", "vit_small_lr3e5": "ViT-S 3e-5", "vit_small": "ViT-S 5e-5",
            "vit_small_lr1e4": "ViT-S 1e-4", "vit_small_lld": "ViT-S 1e-4 + layer decay", "vit": "ViT-B 3e-5"}
val = {r: pd.read_csv(RESULTS / r / "history.csv")["val_macro_f1"].max() for r in vit_runs}

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(list(vit_runs.values()), list(val.values()), color=SERIES[0])
bars[-1].set_color(SERIES[1])  # the selected run
ax.bar_label(bars, fmt="%.3f", fontsize=9)
ax.set_xlim(0.5, 0.72)
ax.set_xlabel("best validation macro-F1 (FER-2013)")
ax.set_title("ViT tuning, one seed per run")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "vit_tuning_val_f1.png", dpi=200, bbox_inches="tight")
plt.show()

/var/folders/dj/9ms80p1d7_xfsf3my3s8tpl00000gn/T/ipykernel_49864/662444730.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figures are written to `results/figures/` and the table to `results/final_comparison.csv`.

Reading notes for the report: ViT-B is the final ViT because it has the best validation macro-F1, by a margin of
0.010 on one seed. Per-run details and caveats are in `results/vit_tuning.md`.